In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from VegasAfterglow import ISM, GaussianJet, Observer, Radiation, Model, Ejecta
from astropy.cosmology import Planck18 as cosmo

# set plt fontfamily
plt.rcParams["font.family"] = "arial"
plt.rcParams["font.size"] = 15


In [2]:
c = 2.99792458e10

: 

In [3]:
Gamma0 = 200

E_iso = 53.03
n_ism = -2.906
theta_c = 0.04646
theta_w = 0.8087
theta_v = 0.2775
p = 2.127
eps_e = -2.869
eps_B = -2.45
xi_e = -1.234
theta_cut = theta_v * 7/6

E_iso = 10**E_iso
n_ism = 10**n_ism
eps_e = 10**eps_e
eps_B = 10**eps_B
xi_e = 10**xi_e

theta_v_list = np.radians([0, 16, 30])

bns1 = {
    'id' :  'bns1',
    'E_iso': E_iso,
    'Gamma0' : Gamma0,
    'theta_c': theta_c,
    'theta_cut': theta_cut,
    'theta_obs_list': theta_v_list,
}

bns2 = {
    'id' : 'bns2',
    'E_iso': E_iso,
    'Gamma0' : Gamma0,
    'theta_c': theta_c,
    'theta_cut': theta_cut,
    'theta_obs_list': theta_v_list,
}

bns3 = {
    'id' : 'bns3',
    'E_iso': E_iso,
    'Gamma0' : Gamma0,
    'theta_c': theta_c,
    'theta_cut': theta_cut,
    'theta_obs_list': theta_v_list,
}

bhns = {
    'id' : 'bhns',
    'E_iso': E_iso/10,
    'Gamma0' : Gamma0,
    'theta_c': np.radians(10),
    'theta_cut': np.radians(40),
    'theta_obs_list': np.radians([0, 16, 50]),
}

z = 0.0099
d_L = cosmo.luminosity_distance(z).to('cm').value

: 

In [4]:
def eps_k_profile_ns(phi, theta):
    eps = 1e51
    return eps

def Gamma0_profile_ns(phi, theta):
    v = 0.3 * c
    Gamma = v_to_Gamma(v)
    return Gamma
    
def eps_k_profile_bh(phi, theta):
    eps0 = 1e50
    theta_cut = np.radians(20)

    theta, phi = np.meshgrid(theta, phi)

    eps = np.zeros_like(theta)
    mask = (theta >= theta_cut) & (theta <= np.pi - theta_cut)
    eps[mask] = eps0
    return eps

def Gamma0_profile_bh(phi, theta):
    v = 0.3 * c
    Gamma0 = v_to_Gamma(v)
    theta_cut = np.radians(20)

    theta, phi = np.meshgrid(theta, phi)

    Gamma = np.ones_like(theta)
    mask = (theta >= theta_cut) & (theta <= np.pi - theta_cut)
    Gamma[mask] = Gamma0
    return Gamma

def v_to_Gamma(v): 
    v = np.asarray(v)
    beta = v / c
    if np.any(beta >= 1):
        raise ValueError("Velocity cannot exceed the speed of light")
    return 1.0 / np.sqrt(1.0 - beta**2)

: 

In [5]:
# import data
def import_data(model_params, theta_obs, path = './data/'):
    gbm_data = pd.read_csv(path + 'gbm_data.csv')
    integral_data = pd.read_csv(path + 'integral_data.csv')
    chandra_data = pd.read_csv(path + 'chandra_data.csv')
    prompt_model = pd.read_csv(path + str(model_params['id']) + '/' + str(int(round(np.degrees(theta_obs), 0))) + '_data.csv')

    data = {
        'gbm_data': gbm_data,
        'integral_data': integral_data,
        'chandra_data': chandra_data,
        'prompt_model': prompt_model
    }

    return data

: 

In [6]:
def add_late_points(t, L, t_min_extra=None, n_extra=50):
    # Use last part of t for extra points
    if t_min_extra is None:
        t_min_extra = t[-2]  # second-to-last point as starting point
    mask = t >= t_min_extra
    t_late = t[mask]
    L_late = L[mask]
    if len(t_late) < 2:
        return t, L  # not enough points to add
    t_extra = np.logspace(np.log10(t_late[0]), np.log10(t_late[-1]), n_extra)
    L_extra = np.interp(np.log10(t_extra), np.log10(t_late), np.log10(L_late))
    L_extra = 10**L_extra
    # Combine with early points
    t_combined = np.concatenate([t[t < t_min_extra], t_extra])
    L_combined = np.concatenate([L[t < t_min_extra], L_extra])
    # Sort
    sort_idx = np.argsort(t_combined)
    return t_combined[sort_idx], L_combined[sort_idx]

: 

In [1]:
from re import X
from turtle import mode


def plot_all(data, model_params, theta_obs):
    gbm_data = data['gbm_data']
    integral_data = data['integral_data']
    chandra_data = data['chandra_data']
    prompt_model = data['prompt_model']
    jet_ag_model = data['jet_ag_model']
    kn_ag_model = data['kn_ag_model']

    t_jet = np.array(prompt_model['jet_t'])
    L_jet_X = np.array(prompt_model['jet_L_X'])
    L_jet_gamma = np.array(prompt_model['jet_L_gamma'])

    t_ag = np.array(jet_ag_model['t'])
    L_jet_ag = np.array(jet_ag_model['L'])
    L_kn_ag = np.array(kn_ag_model['L'])

    # Define global time grid for full curve
    t_prompt = np.geomspace(t_jet.min(), t_ag.min(), 200)
    t_afterglow = np.geomspace(t_ag.min(), 1e9, 200)

    if model_params['id'] in ['bns1', 'bns2']:
        # Interpolate everything on one consistent grid
        t_tot = np.geomspace(1e-3, 1e9, 400)

        jet_L_interp = np.interp(t_tot, t_jet, L_jet_X)
        jet_ag_L_interp = np.interp(t_tot, t_ag, L_jet_ag)
        kn_ag_L_interp = np.interp(t_tot, t_ag, L_kn_ag)

        t_wind = np.array(prompt_model['wind_t'])
        L_wind = np.array(prompt_model['wind_L_X'])
        t_wind, L_wind = add_late_points(t_wind, L_wind, t_min_extra=1e6)
        wind_L_interp = np.interp(t_tot, t_wind, L_wind)

        L_X_tot = jet_L_interp + wind_L_interp + jet_ag_L_interp + kn_ag_L_interp

    else:
        # --- Prompt phase ---
        mask_prompt = t_jet <= t_ag.min()
        t_prompt_full = t_jet[mask_prompt]
        L_prompt_full = L_jet_X[mask_prompt]

        # --- Afterglow phase ---
        jet_L_interp = np.interp(t_afterglow, t_jet, L_jet_X)
        jet_ag_L_interp = np.interp(t_afterglow, t_ag, L_jet_ag)
        kn_ag_L_interp = np.interp(t_afterglow, t_ag, L_kn_ag)
        L_X_afterglow = jet_L_interp + jet_ag_L_interp + kn_ag_L_interp

        # --- Combine ---
        t_tot = np.concatenate([t_prompt_full, t_afterglow])
        L_X_tot = np.concatenate([L_prompt_full, L_X_afterglow])
    
    fig, ax = plt.subplots(figsize=(6, 5))

    plt.plot(t_tot, L_X_tot, color='black', label='Total X-ray', lw=2)

    ax.errorbar(chandra_data['t'], chandra_data['L'], yerr=[chandra_data['L_err_neg'], chandra_data['L_err_pos']], fmt='o', label='GRB 170817A (Chandra)', color='k', ms=4)
    # ax.errorbar(gbm_data['t'], gbm_data['L'], xerr=1, yerr=[gbm_data['L_err_neg'], gbm_data['L_err_pos']], fmt='o', label='GRB 170817A (GBM)', color='grey', ms=0)
    ax.errorbar(
        1, 2.09e46,
        xerr=[[1], [1]],
        yerr=[[0.5e46], [3.27e46]],
        linestyle='none',
        capsize=2,
        ms=4,
        elinewidth=1,
        color='grey',
        label='GRB 170817A (GBM)'
    )
    ax.plot(t_jet, L_jet_X, color='red', label='Prompt', ls='--')
    if model_params['id'] == 'bns1' or model_params['id'] == 'bns2':
        ax.plot(t_wind, L_wind, color='blue', label='Wind', ls='--')

    ax.plot(t_ag, L_jet_ag, color='green', label='Jet Afterglow', ls='--')
    ax.plot(t_ag, L_kn_ag, color='goldenrod', label='Kilonova Afterglow', ls='--')

    ax.plot(t_jet, L_jet_gamma, color='grey', label=r'$\gamma$-ray', ls='--')

    ax.fill_betweenx(
        [1e20, 1e60],
        x1=0,
        x2=1.7e4,
        color='grey',
        alpha=0.15,
    )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(1e-3, 1e9)
    ax.set_ylim(1e33, 1e53)
    ax.set_xlabel(r'$t$ [s]')
    ax.set_ylabel(r'Luminosity [erg/s]')
    if theta_obs == model_params['theta_obs_list'][0]:
        ax.legend(loc='upper right', fontsize=10, frameon=False)

    if model_params['id'] == 'bns1':
        ax.axvline(1e-1, color='grey', ls=':')
        ax.axvline(300, color='grey', ls=':')
        ax.text(1e-2, 5e51, 'HMNS', color='grey', ha='center', va='center', fontsize=14)
        ax.text(5, 5e51, 'SMNS', color='grey', ha='center', va='center', fontsize=14)
        ax.text(2e3, 5e51, 'NS', color='grey', ha='center', va='center', fontsize=14)
        if theta_obs > model_params['theta_cut']:
            ax.axvline(1.33e5, color='k', ls=':')

    if model_params['id'] == 'bns2':
        ax.axvline(1e-1, color='grey', ls=':')
        ax.axvline(300, color='grey', ls=':')
        ax.text(1e-2, 5e51, 'HMNS', color='grey', ha='center', va='center', fontsize=14)
        ax.text(5, 5e51, 'SMNS', color='grey', ha='center', va='center', fontsize=14)
        ax.text(2e3, 5e51, 'BH', color='grey', ha='center', va='center', fontsize=14)
        if theta_obs > model_params['theta_cut']:
            ax.axvline(1.33e5, color='k', ls=':')

    if model_params['id'] == 'bns3':
        ax.axvline(1e-1, color='grey', ls=':')
        ax.axvline(300, color='grey', ls=':')
        ax.text(1e-2, 5e51, '(HMNS)', color='grey', ha='center', va='center', fontsize=14)
        ax.text(5, 5e51, 'BH', color='grey', ha='center', va='center', fontsize=14)
        if theta_obs > model_params['theta_cut']:
            ax.axvline(1.33e5, color='k', ls=':')

    if theta_obs < model_params['theta_c']:
        ax.text(0.5, 1.01, r'Jet zone $({{{}}}\degree)$'.format(int(round(np.degrees(theta_obs), 0))), transform=ax.transAxes,
        ha='center', va='bottom', fontsize=18)
    elif theta_obs < model_params['theta_cut']:
        ax.text(0.5, 1.01, r'Free zone $({{{}}}\degree)$'.format(int(round(np.degrees(theta_obs), 0))), transform=ax.transAxes,
        ha='center', va='bottom', fontsize=18)
    else:
        ax.text(0.5, 1.008, r'Trapped zone $({{{}}}\degree)$'.format(int(round(np.degrees(theta_obs), 0))), transform=ax.transAxes,
        ha='center', va='bottom', fontsize=18)


    ax.tick_params(axis='x', which='both', top=True, direction='in')
    ax.tick_params(axis='y', which='both', right=True, direction='in')

    plt.tight_layout()
    # fig.subplots_adjust(left=0.12, right=0.95, top=0.5, bottom=0.1)
    plt.savefig(str(model_params['id']) + '_' + str(int(round(np.degrees(theta_obs), 0))) + '.pdf', bbox_inches='tight', pad_inches=0.05)
    plt.show()

: 

In [2]:
def run_model(model_params):
    ev_min = 0.3e3
    ev_max = 8e3
    hz_min = ev_min / 4.135667696e-15
    hz_max = ev_max / 4.135667696e-15

    times = np.logspace(-1.8, 11, 1000) 

    theta_c = model_params['theta_c']
    E_iso = model_params['E_iso']
    Gamma0 = model_params['Gamma0']
    # theta_cut = model_params['theta_cut']

    medium = ISM(n_ism=n_ism)

    jet = GaussianJet(theta_c=theta_c, E_iso=E_iso, Gamma0=Gamma0)
    
    if model_params['id'] == 'bns1' or model_params['id'] == 'bns2':
        kn = Ejecta(E_iso=eps_k_profile_ns, Gamma0=Gamma0_profile_ns)
    else:
        kn = Ejecta(E_iso=eps_k_profile_bh, Gamma0=Gamma0_profile_bh)

    rad = Radiation(eps_e=eps_e, eps_B=eps_B, p=p, xi_e=xi_e)

    for theta_obs in model_params['theta_obs_list']:
        obs = Observer(lumi_dist=d_L, z=z, theta_obs=theta_obs)

        jet_model = Model(jet=jet, medium=medium, observer=obs, fwd_rad=rad)
        kn_model = Model(jet=kn, medium=medium, observer=obs, fwd_rad=rad)

        jet_results = jet_model.flux(times, hz_min, hz_max, 200)
        jet_F = jet_results.total
        jet_L = jet_F * 4 * np.pi * d_L**2 * (1 + z)

        kn_results = kn_model.flux(times, hz_min, hz_max, 200)
        kn_F = kn_results.total 
        kn_L = kn_F * 4 * np.pi * d_L**2 * (1 + z)

        jet_ag_model = pd.DataFrame({'t': times, 'L': jet_L})
        kn_ag_model = pd.DataFrame({'t': times, 'L': kn_L})

        data = import_data(model_params, theta_obs)
        data['jet_ag_model'] = jet_ag_model
        data['kn_ag_model'] = kn_ag_model
        
        plot_all(data, model_params, theta_obs)

    return jet_ag_model, kn_ag_model

: 

In [ ]:
for model_params in [bns1, bns2, bns3, bhns]:
    jet_ag_model, kn_ag_model = run_model(model_params)



NameError: name 'bns1' is not defined

: 